# Claims Validation tests

### Key Functions:
1. **raw_source_ndjson_data_stats**: Computes statistics for NDJSON files within a root directory.
2. **get_bronze_clinical_fhir_statistics**: Computes statistics on the number of records and distinct ids per resource type in the Bronze ClinicalFHIR Delta table.
3. **get_silver_fhir_statistics**: Computes statistics on the number of records and distinct IDs per resource type in the Silver ClinicalFHIR Delta table.
4. **compare_clinical_fhir_to_lakehouse**: Compares ClinicalFHIR statistics to Lakehouse statistics to identify discrepancies.
5. **compare_raw_source_to_bronze**: Compares statistics between raw NDJSON source data and the Bronze ClinicalFHIR table to identify discrepancies.

In [ ]:
workspace_id = ""
bronze_lakehouse_id = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""
deployment_environment = ""
silver_lakehouse_id = ""

In [ ]:
bronze_delta_table_path = f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_id}/Tables/ClinicalFhir"

In [ ]:
from pyspark.sql.functions import input_file_name, col, count, countDistinct, coalesce, lit
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
from concurrent.futures import ThreadPoolExecutor
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner

class ClinicalFoundationsIngestionTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None, silver_lakehouse_id = None):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id
        self.silver_lakehouse_id = silver_lakehouse_id
    def raw_source_ndjson_data_stats(self, root_source_data_path) -> DataFrame:
        """
        Function to get the count of 'ExplanationOfBenefit' records in NDJSON files within the root source path.
        
        :param root_source_data_path: Root path to source data.
        :return: A DataFrame containing the count of 'ExplanationOfBenefit' records.
        """
        files = mssparkutils.fs.ls(root_source_data_path)
        if not files:  # If the folder is empty, return an empty DataFrame

            return self.spark.createDataFrame([], schema="resourceType STRING, record_count INT")
        # Use Spark to recursively read all NDJSON files from the root source path
        df = self.spark.read.format("json").option("recursiveFileLookup", "true").load(root_source_data_path)
        df = df.filter(input_file_name().endswith('.ndjson'))
        
        # Filter for 'ExplanationOfBenefit' resourceType
        eob_df = df.filter(col("resourceType") == "ExplanationOfBenefit")

        resource_counts_df = eob_df.groupBy("resourceType").agg(
                count("*").alias("record_count")
            )
        
        return resource_counts_df
    
    def get_bronze_clinical_fhir_statistics(self,delta_table_path) -> DataFrame:
        """
        Function to get statistics on the number of records per resourceType and the number of distinct ids per resourceType.
        
        :param delta_table_path: Path to the Delta table.
        :return: A DataFrame containing resourceType, total count, and distinct id count.
        """
        # Read the entire Delta table
        df = self.spark.read.format("delta").load(delta_table_path)
        
        # Group by resourceType and calculate counts and distinct counts
        bronze_stats_df = df.filter(col("resourceType") == "ExplanationOfBenefit") \
                            .groupBy("resourceType") \
                            .agg(
                            count("*").alias("count"),
                            countDistinct("id").alias("distinct_count")
                            )
        return bronze_stats_df

    def get_silver_fhir_statistics(self,delta_table_path) -> DataFrame:
        """
        Function to get statistics on the number of records per resourceType and the number of distinct ids per resourceType.
        
        :param delta_table_path: Path to the Delta table.
        :return: A DataFrame containing resourceType, total count, and distinct id count.
        """
        # Read the entire Delta table
        df = self.spark.read.format("delta").load(delta_table_path)
        
        # Group by resourceType and calculate counts and distinct counts
        bronze_stats_df = df.filter(col("resourceType") == "ExplanationOfBenefit") \
                            .groupBy("resourceType") \
                            .agg(
                            count("*").alias("count"),
                            countDistinct("id").alias("silver_record_count")
                            )
        return bronze_stats_df

    def compare_record_counts(self,  clinical_df,silver_df) -> DataFrame:
        """
        Function to compare record counts of ExplanationOfBenefit between Silver ClinicalFHIR and bronze dataset.

        :param silver_df: DataFrame containing resourceType and silver_record_count.
        :param clinical_df: DataFrame containing resourceType and record count.
        :return: A DataFrame containing resourceType, silver record count, other record count, and discrepancy.
        """
        # Join the DataFrames on resourceType (only ExplanationOfBenefit is considered)
        comparison_df = silver_df.join(
            clinical_df,
            silver_df.resourceType == clinical_df.resourceType,
            "outer"
        ).select(
            coalesce(silver_df.resourceType, clinical_df.resourceType).alias("resource"),
            coalesce(silver_df["silver_record_count"], lit(0)).alias("silver_record_count"),
            coalesce(clinical_df["distinct_count"], lit(0)).alias("clinical_record_count"),
            (coalesce(silver_df["silver_record_count"], lit(0)) - coalesce(clinical_df["distinct_count"], lit(0))).alias("record_discrepancy")
        )

        # Identify discrepancies
        discrepancies_df = comparison_df.filter(col("record_discrepancy") != 0)

        # Display discrepancies if found
        if discrepancies_df.count() > 0:
            print("\n Discrepancies found between Silver ClinicalFHIR and the Silver dataset:")
            display(discrepancies_df)
        else:
            print("\n No discrepancies found.")

        return comparison_df


    def compare_raw_source_to_bronze(self, raw_source_df, bronze_df) -> DataFrame:
        """
        Function to compare the distinct count of records in the raw source NDJSON data with the count of records
        in the bronze ClinicalFHIR table.
        
        :param raw_source_df: DataFrame containing raw source NDJSON statistics.
        :param bronze_df: DataFrame containing Bronze ClinicalFHIR statistics.
        :return: A DataFrame containing resourceType, raw source record count, bronze record count, and the difference.
        """
        # Join the DataFrames on resourceType
        comparison_df = raw_source_df.join(
            bronze_df, 
            raw_source_df.resourceType == bronze_df.resourceType, 
            "outer"
        ).select(
            coalesce(raw_source_df.resourceType, bronze_df.resourceType).alias("resource"),
            coalesce(col("record_count"), lit(0)).alias("raw_source_record_count"),
            coalesce(col("count"), lit(0)).alias("bronze_record_count"),
            (coalesce(col("record_count"), lit(0)) - coalesce(col("count"), lit(0))).alias("discrepancy_count")
        )
        
        # Check for discrepancies and print details if any
        discrepancies_df = comparison_df.filter(col("discrepancy_count") != 0)
        if discrepancies_df.count() > 0:
            print("Discrepancies found between raw source NDJSON records and Bronze ClinicalFHIR table:")
            display(discrepancies_df)
        else:
            print("No discrepancies found between raw source NDJSON records and Bronze ClinicalFHIR table.")
        
        return comparison_df
    
    def test_raw_to_bronze_data_ingestion(self):
        
        root_ndjson_data_path = f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/Process/Clinical/FHIR-NDJSON/FHIR-HDS/"
        source_data_stats_df = self.raw_source_ndjson_data_stats(root_ndjson_data_path).cache()
        bronze_statistics_df = self.get_bronze_clinical_fhir_statistics(bronze_delta_table_path).cache()
        source_to_bronze_comparison_df = self.compare_raw_source_to_bronze(source_data_stats_df, bronze_statistics_df)
        
        # Iterate over the rows and perform the checks
        rows = source_to_bronze_comparison_df.collect()
        for row in rows:
            resource = row['resource']
            discrepancy_count = row['discrepancy_count']

    def test_bronze_to_silver_data_ingestion(self):

        bronze_delta_table_path = f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Tables/ClinicalFhir"
        silver_delta_table_path = f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.silver_lakehouse_id}/Tables/ExplanationOfBenefit"
        bronze_statistics_df = self.get_bronze_clinical_fhir_statistics(bronze_delta_table_path).cache()
        silver_stats_df = self.get_silver_fhir_statistics(silver_delta_table_path).cache()

        comparison_df = self.compare_record_counts(bronze_statistics_df, silver_stats_df)
        display(comparison_df)

        rows = comparison_df.collect()

        # Iterate over the rows and perform the checks
        for row in rows:
            resource = row['resource']
            discrepancy_count = row['record_discrepancy']
        
            

def run_tests_and_write_output(spark):
    
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(ClinicalFoundationsIngestionTests)

    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id
        test.bronze_lakehouse_id = bronze_lakehouse_id
        test.silver_lakehouse_id = silver_lakehouse_id

    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream, verbosity=3).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')

    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output

report = run_tests_and_write_output(spark)